# BanglaBERT Fine-Tuning for AI-Generated Text Detection

This notebook fine-tunes BanglaBERT to classify Bangla text as human-written (0) or AI-generated (1).

**Overfitting Prevention Strategies Used:**
- `weight_decay`: L2 regularization on model weights
- `warmup_ratio`: Gradual learning rate warmup to avoid large early updates
- `EarlyStoppingCallback`: Stops training when validation F1 stops improving
- Lower `learning_rate` values: Reduces the chance of aggressive weight updates
- `hidden_dropout_prob` & `attention_probs_dropout_prob`: Dropout inside BanglaBERT's layers (set via model config)
- `classifier_dropout`: Dropout on the final classification head


In [2]:
# ─────────────────────────────────────────────
# CELL 1: Import all required libraries
# ─────────────────────────────────────────────

import numpy as np
import pandas as pd
import torch
import os
import shutil
import gc
import json

from pathlib import Path
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    AutoConfig,           # NEW: needed to set dropout in model config
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback
)
from sklearn.metrics import (
    accuracy_score, precision_score,
    recall_score, f1_score,
    roc_auc_score, average_precision_score, brier_score_loss
)
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix
from sklearn.calibration import calibration_curve

print("All libraries imported successfully.")
print(f"PyTorch version: {torch.__version__}")
print(f"GPU available: {torch.cuda.is_available()}")


All libraries imported successfully.
PyTorch version: 2.10.0+cu128
GPU available: True


In [3]:
# ─────────────────────────────────────────────
# CELL 2: Setup Kaggle paths & locate data files
# ─────────────────────────────────────────────

print("Running BanglaBERT fine-tuning notebook on Kaggle")
print("Kaggle input root   : /kaggle/input")
print("Kaggle working root : /kaggle/working")

KAGGLE_INPUT_ROOT = Path("/kaggle/input")
PREFERRED_DATASET_FOLDER = None  # Set to a specific folder name if needed

# Automatically locate the split CSV files inside /kaggle/input
if PREFERRED_DATASET_FOLDER is not None:
    split_dir = KAGGLE_INPUT_ROOT / PREFERRED_DATASET_FOLDER
else:
    train_candidates = list(KAGGLE_INPUT_ROOT.rglob("train_split.csv"))
    val_candidates   = list(KAGGLE_INPUT_ROOT.rglob("val_split.csv"))
    test_candidates  = list(KAGGLE_INPUT_ROOT.rglob("test_split.csv"))

    if not train_candidates or not val_candidates or not test_candidates:
        raise FileNotFoundError(
            "Could not find train_split.csv, val_split.csv, and test_split.csv inside /kaggle/input. "
            "Attach your shared-splits dataset first."
        )

    split_dir = train_candidates[0].parent

TRAIN_PATH = split_dir / "train_split.csv"
VAL_PATH   = split_dir / "val_split.csv"
TEST_PATH  = split_dir / "test_split.csv"

print("Using split folder:", split_dir)
print("TRAIN_PATH:", TRAIN_PATH)
print("VAL_PATH  :", VAL_PATH)
print("TEST_PATH :", TEST_PATH)


Running BanglaBERT fine-tuning notebook on Kaggle
Kaggle input root   : /kaggle/input
Kaggle working root : /kaggle/working
Using split folder: /kaggle/input/datasets/prohorpaul/ai-paraphrased-text
TRAIN_PATH: /kaggle/input/datasets/prohorpaul/ai-paraphrased-text/train_split.csv
VAL_PATH  : /kaggle/input/datasets/prohorpaul/ai-paraphrased-text/val_split.csv
TEST_PATH : /kaggle/input/datasets/prohorpaul/ai-paraphrased-text/test_split.csv


In [4]:
# ─────────────────────────────────────────────
# CELL 3: Load datasets and inspect
# ─────────────────────────────────────────────

train_df = pd.read_csv(TRAIN_PATH)
val_df   = pd.read_csv(VAL_PATH)
test_df  = pd.read_csv(TEST_PATH)

# Reset index to avoid any index mismatches
train_df = train_df.reset_index(drop=True)
val_df   = val_df.reset_index(drop=True)
test_df  = test_df.reset_index(drop=True)

print("Dataset shapes:")
print("  Train shape      :", train_df.shape)
print("  Validation shape :", val_df.shape)
print("  Test shape       :", test_df.shape)


Dataset shapes:
  Train shape      : (10326, 3)
  Validation shape : (1476, 3)
  Test shape       : (2952, 3)


In [5]:
# ─────────────────────────────────────────────
# CELL 4: Inspect class balance and null values
# ─────────────────────────────────────────────

print("=== Label Distribution ===")
print("Train:")
print(train_df["label"].value_counts())
print("\nValidation:")
print(val_df["label"].value_counts())
print("\nTest:")
print(test_df["label"].value_counts())

print("\n=== Null Check ===")
for name, df in [("Train", train_df), ("Val", val_df), ("Test", test_df)]:
    nulls = df[["text", "label"]].isnull().sum()
    print(f"  {name}: {nulls.to_dict()}")

print("\n=== Sample Text (first train row) ===")
print(train_df.loc[0, "text"][:500])


=== Label Distribution ===
Train:
label
0    5163
1    5163
Name: count, dtype: int64

Validation:
label
0    738
1    738
Name: count, dtype: int64

Test:
label
0    1476
1    1476
Name: count, dtype: int64

=== Null Check ===
  Train: {'text': 0, 'label': 0}
  Val: {'text': 0, 'label': 0}
  Test: {'text': 0, 'label': 0}

=== Sample Text (first train row) ===
রাজধানীর ক্যান্টনমেন্ট থানা এলাকায় গতকাল শুক্রবার সকালে লিফট থেকে পড়ে আবু সুফিয়ান (৪৫) নামের এক নিরাপত্তাকর্মী মারা গেছেন। এ ছাড়া খিলগাঁওয়ে পুকুরে ডুবে এক শিশুর মৃত্যু হয়েছে। ক্যান্টনমেন্ট থানার উপপরিদর্শক (এসআই) লিয়াকত আলী সাংবাদিকদের জানান, নিহত সুফিয়ান বনানীর ডিওএইচএস এলাকার ৬ নম্বর সড়কের একটি বাড়ির নিরাপত্তাকর্মী ছিলেন। গতকাল সকাল সাড়ে আটটার দিকে তিনি ছয়তলা থেকে লিফটে করে নিচে নামছিলেন। মাঝখানে এসে লিফটটি হঠাৎ বন্ধ হয়ে যায়। আশপাশের লোকজন অনেক চেষ্টা করে কোনোমতে লিফটের দরজা ফাঁক করে সুফিয়ানকে


In [6]:
# ─────────────────────────────────────────────
# CELL 5: Keep only relevant columns
# ─────────────────────────────────────────────

# We only need 'text' and 'label' columns for training
train_df = train_df[["text", "label"]].copy()
val_df   = val_df[["text", "label"]].copy()
test_df  = test_df[["text", "label"]].copy()

print("Columns kept: ['text', 'label']")
display(train_df.head())


Columns kept: ['text', 'label']


,text,label
0,রাজধানীর ক্যান্টনমেন্ট থানা এলাকায় গতকাল শুক্র...,0
1,বাংলাদেশে নিযুক্ত মার্কিন রাষ্ট্রদূত ড্যান ডব্...,0
2,নীলফামারীর কিশোরগঞ্জে বিদ্যুতের তারে জড়িয়ে এ...,0
3,শেরপুরের ঝিনাইগাতী উপজেলায় গতকাল শুক্রবার সড়ক ...,0
4,ফরিদপুরে সাপের কামড়ে দুই শিক্ষার্থী নিহত হয়ে...,0


In [7]:
# ─────────────────────────────────────────────
# CELL 6: Load BanglaBERT tokenizer
# ─────────────────────────────────────────────

MODEL_NAME = "csebuetnlp/banglabert"  # Pre-trained BanglaBERT from HuggingFace
MAX_LENGTH = 256  # Max token length — BanglaBERT supports up to 512 tokens

# Load the tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print(f"Tokenizer loaded: {MODEL_NAME}")


config.json:   0%|          | 0.00/586 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/119 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Tokenizer loaded: csebuetnlp/banglabert


In [8]:
# ─────────────────────────────────────────────
# CELL 7: Convert DataFrames to HuggingFace Datasets
# ─────────────────────────────────────────────

# HuggingFace Dataset objects are more efficient and
# integrate directly with the Trainer API
train_dataset = Dataset.from_pandas(train_df)
val_dataset   = Dataset.from_pandas(val_df)
test_dataset  = Dataset.from_pandas(test_df)

print("Datasets created:")
print("  Train:", train_dataset)
print("  Val  :", val_dataset)
print("  Test :", test_dataset)


Datasets created:
  Train: Dataset({
    features: ['text', 'label'],
    num_rows: 10326
})
  Val  : Dataset({
    features: ['text', 'label'],
    num_rows: 1476
})
  Test : Dataset({
    features: ['text', 'label'],
    num_rows: 2952
})


In [9]:
# ─────────────────────────────────────────────
# CELL 8: Tokenize all datasets
# ─────────────────────────────────────────────

def tokenize_function(example):
    """
    Tokenizes a batch of text using BanglaBERT's tokenizer.
    - truncation=True: Clips text longer than MAX_LENGTH
    - padding='max_length': Pads shorter sequences to MAX_LENGTH
    - max_length=MAX_LENGTH: Sets the fixed token length
    """
    return tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=MAX_LENGTH
    )

# Apply tokenization to all splits (batched=True for speed)
train_dataset = train_dataset.map(tokenize_function, batched=True)
val_dataset   = val_dataset.map(tokenize_function, batched=True)
test_dataset  = test_dataset.map(tokenize_function, batched=True)

# Remove the raw text column — the model only needs token IDs
train_dataset = train_dataset.remove_columns(["text"])
val_dataset   = val_dataset.remove_columns(["text"])
test_dataset  = test_dataset.remove_columns(["text"])

# Set the format to PyTorch tensors
train_dataset.set_format("torch")
val_dataset.set_format("torch")
test_dataset.set_format("torch")

print("Tokenization complete.")
print("Sample keys in train_dataset:", train_dataset.column_names)


Map:   0%|          | 0/10326 [00:00<?, ? examples/s]

Map:   0%|          | 0/1476 [00:00<?, ? examples/s]

Map:   0%|          | 0/2952 [00:00<?, ? examples/s]

Tokenization complete.
Sample keys in train_dataset: ['label', 'input_ids', 'token_type_ids', 'attention_mask']


In [10]:
# ─────────────────────────────────────────────
# CELL 9: Define evaluation metrics
# ─────────────────────────────────────────────

def compute_metrics(eval_pred):
    """
    Called by the Trainer at the end of each evaluation step.
    Takes raw logits and true labels, returns a metrics dictionary.
    """
    logits, labels = eval_pred
    # Convert logits to class predictions (0 or 1)
    preds = np.argmax(logits, axis=1)

    return {
        "accuracy" : accuracy_score(labels, preds),
        "precision": precision_score(labels, preds),
        "recall"   : recall_score(labels, preds),
        "f1"       : f1_score(labels, preds)
    }


In [11]:
# ─────────────────────────────────────────────
# CELL 10: Define hyperparameter search grid
#
# OVERFITTING NOTE — What each parameter does:
#
# lr (learning_rate):
#   Lower LR (1e-5) → smaller weight updates → less risk of overfitting.
#   2e-5 is still safe for BERT-style models but may overfit faster.
#
# weight_decay:
#   L2 regularization on all weights (except biases & LayerNorm).
#   Penalizes large weights → reduces overfitting.
#   0.01 is the standard value for transformers.
#
# warmup_ratio:
#   Fraction of training steps for LR warmup.
#   Prevents large gradient updates at the start → more stable training.
#   0.1 means 10% of training steps are warmup.
#
# hidden_dropout_prob & attention_probs_dropout_prob:
#   Dropout applied inside the transformer layers.
#   Set via model config (see training loop below).
#   Higher dropout → more regularization.
#
# classifier_dropout:
#   Dropout applied just before the final classification layer.
#   Prevents the classifier head from memorizing training patterns.
#
# EarlyStoppingCallback (patience=2):
#   Stops training if validation F1 doesn't improve for 2 epochs.
#   This is the most direct defense against overfitting.
# ─────────────────────────────────────────────

param_grid = [
    # Config 1: Low LR, small batch, moderate regularization
    {'lr': 1e-5, 'batch_size': 8,  'weight_decay': 0.01, 'warmup_ratio': 0.1,
     'hidden_dropout': 0.1, 'attn_dropout': 0.1, 'classifier_dropout': 0.1, 'epochs': 5},

    # Config 2: Slightly higher LR, larger batch, stronger regularization
    {'lr': 2e-5, 'batch_size': 16, 'weight_decay': 0.01, 'warmup_ratio': 0.1,
     'hidden_dropout': 0.2, 'attn_dropout': 0.1, 'classifier_dropout': 0.2, 'epochs': 5},

    # Config 3: Low LR, large batch, high dropout — most conservative
    {'lr': 1e-5, 'batch_size': 16, 'weight_decay': 0.01, 'warmup_ratio': 0.06,
     'hidden_dropout': 0.2, 'attn_dropout': 0.2, 'classifier_dropout': 0.2, 'epochs': 5},

    # Config 4: Higher LR, small batch, minimal dropout — more aggressive training
    {'lr': 2e-5, 'batch_size': 8,  'weight_decay': 0.01, 'warmup_ratio': 0.1,
     'hidden_dropout': 0.1, 'attn_dropout': 0.1, 'classifier_dropout': 0.1, 'epochs': 5},
]

print(f"Total hyperparameter configs to try: {len(param_grid)}")
for i, p in enumerate(param_grid):
    print(f"  Config {i+1}: {p}")


Total hyperparameter configs to try: 4
  Config 1: {'lr': 1e-05, 'batch_size': 8, 'weight_decay': 0.01, 'warmup_ratio': 0.1, 'hidden_dropout': 0.1, 'attn_dropout': 0.1, 'classifier_dropout': 0.1, 'epochs': 5}
  Config 2: {'lr': 2e-05, 'batch_size': 16, 'weight_decay': 0.01, 'warmup_ratio': 0.1, 'hidden_dropout': 0.2, 'attn_dropout': 0.1, 'classifier_dropout': 0.2, 'epochs': 5}
  Config 3: {'lr': 1e-05, 'batch_size': 16, 'weight_decay': 0.01, 'warmup_ratio': 0.06, 'hidden_dropout': 0.2, 'attn_dropout': 0.2, 'classifier_dropout': 0.2, 'epochs': 5}
  Config 4: {'lr': 2e-05, 'batch_size': 8, 'weight_decay': 0.01, 'warmup_ratio': 0.1, 'hidden_dropout': 0.1, 'attn_dropout': 0.1, 'classifier_dropout': 0.1, 'epochs': 5}


In [ ]:
# ─────────────────────────────────────────────
# CELL 11: Training loop with overfitting prevention
#
# Key anti-overfitting additions vs original:
#   1. warmup_ratio  — gradual LR warm-up
#   2. Dropout via AutoConfig — regularizes hidden layers
#   3. classifier_dropout — regularizes the classification head
#   4. EarlyStoppingCallback — halts training early if val F1 stops improving
#   5. No model/checkpoint saving (as requested)
# ─────────────────────────────────────────────




# ─────────────────────────────────────────────
# CELL 11: Training loop with early stopping by validation loss
# ─────────────────────────────────────────────

best_val_loss = float("inf")
best_config   = None
best_trainer  = None
best_run_dir  = None
results_all   = []

# Temporary directory used only while training
RUNS_DIR = "/kaggle/working/banglabert_runs"
os.makedirs(RUNS_DIR, exist_ok=True)

for i, params in enumerate(param_grid):
    print(f"\n{'='*60}")
    print(f"Running config {i+1}/{len(param_grid)}: {params}")
    print('='*60)

    run_dir = f"{RUNS_DIR}/run_{i}"

    # Load model config with dropout values
    config = AutoConfig.from_pretrained(
        MODEL_NAME,
        num_labels=2,
        hidden_dropout_prob=params["hidden_dropout"],
        attention_probs_dropout_prob=params["attn_dropout"],
        classifier_dropout=params["classifier_dropout"],
    )

    # Load BanglaBERT with modified config
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        config=config,
    )

    # Training arguments
    training_args = TrainingArguments(
        output_dir=run_dir,

        # Evaluation and logging
        eval_strategy="epoch",
        logging_strategy="epoch",

        # Save temporarily so best checkpoint can be restored
        save_strategy="epoch",
        save_total_limit=1,
        save_only_model=True,
        load_best_model_at_end=True,

        # Optimization
        learning_rate=params["lr"],
        per_device_train_batch_size=params["batch_size"],
        per_device_eval_batch_size=params["batch_size"],
        num_train_epochs=params["epochs"],

        # Anti-overfitting
        weight_decay=params["weight_decay"],
        warmup_ratio=params["warmup_ratio"],

        # Best model selection by validation loss
        metric_for_best_model="eval_loss",
        greater_is_better=False,

        report_to="none"
    )

    # Build trainer
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        compute_metrics=compute_metrics,
        callbacks=[
            EarlyStoppingCallback(early_stopping_patience=2)
        ]
    )

    # Train
    trainer.train()

    # Evaluate on validation set
    eval_result = trainer.evaluate()
    current_val_loss = eval_result["eval_loss"]
    current_f1 = eval_result["eval_f1"]

    print(f"Validation Loss: {current_val_loss:.4f}")
    print(f"Validation F1   : {current_f1:.4f}")

    # Store results for comparison later
    results_all.append({
        "lr"                : params["lr"],
        "batch_size"        : params["batch_size"],
        "weight_decay"      : params["weight_decay"],
        "warmup_ratio"      : params["warmup_ratio"],
        "hidden_dropout"    : params["hidden_dropout"],
        "attn_dropout"      : params["attn_dropout"],
        "classifier_dropout": params["classifier_dropout"],
        "epochs"            : params["epochs"],
        "Val Loss"          : current_val_loss,
        "Val F1"            : current_f1
    })

    # Track best configuration using validation loss
    if current_val_loss < best_val_loss:
        # Remove previous best run directory if it exists
        if best_run_dir is not None and os.path.exists(best_run_dir):
            shutil.rmtree(best_run_dir)

        # Free old best trainer/model from memory if needed
        if best_trainer is not None:
            del best_trainer
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

        best_val_loss = current_val_loss
        best_config = params.copy()
        best_trainer = trainer
        best_run_dir = run_dir

        print(f"✓ New best config found! Val Loss = {best_val_loss:.4f}")

    else:
        # Not best, so remove this run and free memory
        if os.path.exists(run_dir):
            shutil.rmtree(run_dir)

        del trainer
        del model
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

        print("✗ Not best. Freed memory.")

print(f"\n{'='*60}")
print(f"Best config: {best_config}")
print(f"Best Val Loss: {best_val_loss:.4f}")


Running config 1/4: {'lr': 1e-05, 'batch_size': 8, 'weight_decay': 0.01, 'warmup_ratio': 0.1, 'hidden_dropout': 0.1, 'attn_dropout': 0.1, 'classifier_dropout': 0.1, 'epochs': 5}


pytorch_model.bin:   0%|          | 0.00/443M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/443M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

ElectraForSequenceClassification LOAD REPORT from: csebuetnlp/banglabert
Key                                               | Status     | 
--------------------------------------------------+------------+-
discriminator_predictions.dense_prediction.weight | UNEXPECTED | 
discriminator_predictions.dense_prediction.bias   | UNEXPECTED | 
electra.embeddings.position_ids                   | UNEXPECTED | 
discriminator_predictions.dense.bias              | UNEXPECTED | 
discriminator_predictions.dense.weight            | UNEXPECTED | 
classifier.out_proj.weight                        | MISSING    | 
classifier.dense.weight                           | MISSING    | 
classifier.dense.bias                             | MISSING    | 
classifier.out_proj.bias                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoi

Epoch,Training Loss,Validation Loss


In [ ]:
# ─────────────────────────────────────────────
# CELL 12: Summarize all hyperparameter results
# ─────────────────────────────────────────────

results_df = (
    pd.DataFrame(results_all)
      .sort_values("Val F1", ascending=False)
      .reset_index(drop=True)
)

print("All configs ranked by Validation F1:")
display(results_df)

# Save results to CSV for reference
RESULTS_DIR = Path("/kaggle/working/banglabert_outputs")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

results_df.to_csv(RESULTS_DIR / "banglabert_tuning_results.csv", index=False)

with open(RESULTS_DIR / "banglabert_best_config.json", "w") as f:
    json.dump(best_config, f, indent=2)

print(f"\nResults saved to: {RESULTS_DIR}")


In [ ]:
# ─────────────────────────────────────────────
# CELL 13: Run inference on the test set
# ─────────────────────────────────────────────

# Use the best trainer (which holds the best model weights)
# to generate predictions on the held-out test set
pred_output = best_trainer.predict(test_dataset)

# pred_output.predictions → raw logits (shape: [n_samples, 2])
# pred_output.label_ids   → true labels
y_pred = np.argmax(pred_output.predictions, axis=1)  # Hard class predictions
y_prob = torch.softmax(
    torch.tensor(pred_output.predictions), dim=1
)[:, 1].numpy()  # Probability of class 1 (AI-generated)
y_true = pred_output.label_ids

print(f"Predictions generated for {len(y_true)} test samples.")


In [ ]:
# ─────────────────────────────────────────────
# CELL 14: Define and run full evaluation
# ─────────────────────────────────────────────

def evaluate_predictions(y_true, y_pred, y_prob=None, model_name="BanglaBERT", split_name="Test"):
    """
    Computes and prints a comprehensive set of classification metrics.

    Parameters:
    - y_true      : Ground truth labels
    - y_pred      : Predicted class labels
    - y_prob      : Predicted probabilities for class 1 (optional)
    - model_name  : Label for display
    - split_name  : 'Train', 'Val', or 'Test'

    Metrics computed:
    - Accuracy  : Overall correct predictions
    - Precision : Of all predicted positives, how many are truly positive?
    - Recall    : Of all actual positives, how many did we catch?
    - F1 Score  : Harmonic mean of Precision and Recall
    - ROC-AUC   : Area under the ROC curve (uses probabilities)
    - AP Score  : Average Precision — key metric for your thesis
    - Brier     : Mean squared error between probability and true label
    """
    acc  = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred)
    rec  = recall_score(y_true, y_pred)
    f1   = f1_score(y_true, y_pred)

    result = {
        "Split"    : split_name,
        "Model"    : model_name,
        "Accuracy" : acc,
        "Precision": prec,
        "Recall"   : rec,
        "F1 Score" : f1
    }

    if y_prob is not None:
        result["ROC-AUC"]    = roc_auc_score(y_true, y_prob)
        result["AP Score"]   = average_precision_score(y_true, y_prob)
        result["Brier Score"]= brier_score_loss(y_true, y_prob)

    print(f"\n{'─'*40}")
    print(f"  {split_name} Results — {model_name}")
    print(f"{'─'*40}")
    print(f"  Accuracy  : {acc*100:.2f}%")
    print(f"  Precision : {prec*100:.2f}%")
    print(f"  Recall    : {rec*100:.2f}%")
    print(f"  F1 Score  : {f1*100:.2f}%")
    if y_prob is not None:
        print(f"  ROC-AUC   : {result['ROC-AUC']*100:.2f}%")
        print(f"  AP Score  : {result['AP Score']*100:.2f}%")
        print(f"  Brier     : {result['Brier Score']:.4f}")

    return result


# Evaluate on the test set
result_banglabert = evaluate_predictions(
    y_true, y_pred,
    y_prob=y_prob,
    model_name="BanglaBERT",
    split_name="Test"
)


In [ ]:
# ─────────────────────────────────────────────
# CELL 15: Save predictions and outputs
# ─────────────────────────────────────────────

from pathlib import Path

OUTPUT_DIR = Path("/kaggle/working/banglabert_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Save raw prediction arrays for downstream use (e.g., combining with LightGBM)
np.save(OUTPUT_DIR / "banglabert_y_true.npy", y_true)
np.save(OUTPUT_DIR / "banglabert_y_pred.npy", y_pred)
np.save(OUTPUT_DIR / "banglabert_y_prob.npy", y_prob)

# Also save a human-readable CSV
pd.DataFrame({
    "y_true": y_true,
    "y_pred": y_pred,
    "y_prob": y_prob
}).to_csv(OUTPUT_DIR / "banglabert_test_outputs.csv", index=False)

print("Saved prediction files:")
print(f"  {OUTPUT_DIR / 'banglabert_y_true.npy'}")
print(f"  {OUTPUT_DIR / 'banglabert_y_pred.npy'}")
print(f"  {OUTPUT_DIR / 'banglabert_y_prob.npy'}")
print(f"  {OUTPUT_DIR / 'banglabert_test_outputs.csv'}")


In [ ]:
# ─────────────────────────────────────────────
# CELL 16: Training vs Validation Loss Plot
#
# This is the KEY overfitting diagnostic:
#   - If val_loss keeps decreasing with train_loss → good fit
#   - If val_loss rises while train_loss falls → overfitting
#   - If both losses plateau high → underfitting
# ─────────────────────────────────────────────

log_history = best_trainer.state.log_history

train_loss = []
val_loss   = []

for log in log_history:
    # Training loss logs have 'loss' but NOT 'eval_loss'
    if "loss" in log and "eval_loss" not in log:
        train_loss.append(log["loss"])
    # Validation logs have 'eval_loss'
    if "eval_loss" in log:
        val_loss.append(log["eval_loss"])

print(f"Training Loss   : {[round(l,4) for l in train_loss]}")
print(f"Validation Loss : {[round(l,4) for l in val_loss]}")

# ── Plot ──
plt.figure(figsize=(8, 5))
plt.plot(range(1, len(train_loss)+1), train_loss, marker='o', label="Training Loss")
plt.plot(range(1, len(val_loss)+1),   val_loss,   marker='s', label="Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training vs Validation Loss\n(divergence = overfitting)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
# ─────────────────────────────────────────────
# CELL 17: Confusion Matrix
# ─────────────────────────────────────────────

cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(6, 5))
sns.heatmap(
    cm, annot=True, fmt="d", cmap="Blues",
    xticklabels=["Human (0)", "AI (1)"],
    yticklabels=["Human (0)", "AI (1)"]
)
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.title("Confusion Matrix — BanglaBERT (Test Set)")
plt.tight_layout()
plt.show()

# Interpretation:
# Top-left  = True Negatives  (correctly predicted Human)
# Top-right = False Positives (Human predicted as AI)
# Bot-left  = False Negatives (AI predicted as Human)
# Bot-right = True Positives  (correctly predicted AI)


In [ ]:
# ─────────────────────────────────────────────
# CELL 18: Calibration Curve
#
# Checks whether the model's predicted probabilities are reliable.
# A well-calibrated model: if it says 80% probability, ~80% of those
# examples should actually be positive.
# ─────────────────────────────────────────────

def plot_calibration_curve(y_true, y_prob, model_name="Model", n_bins=10):
    prob_true, prob_pred = calibration_curve(
        y_true, y_prob, n_bins=n_bins, strategy="uniform"
    )

    plt.figure(figsize=(6, 6))
    plt.plot(prob_pred, prob_true, marker="o", label=model_name)
    plt.plot([0, 1], [0, 1], linestyle="--", label="Perfect Calibration")
    plt.xlabel("Mean Predicted Probability")
    plt.ylabel("Fraction of Positives")
    plt.title(f"Calibration Curve — {model_name}")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

    calib_df = pd.DataFrame({
        "Mean Predicted Probability": prob_pred,
        "Fraction of Positives"     : prob_true
    })
    display(calib_df)


plot_calibration_curve(y_true, y_prob, model_name="BanglaBERT", n_bins=5)


In [ ]:
import os
import gc
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import matplotlib.pyplot as plt

from sklearn.calibration import calibration_curve
from sklearn.metrics import brier_score_loss, log_loss, f1_score, accuracy_score, precision_score, recall_score

# =========================================================
# 1) Get validation and test logits from best BanglaBERT model
# =========================================================
val_output = best_trainer.predict(val_dataset)
test_output = best_trainer.predict(test_dataset)

val_logits = val_output.predictions
val_labels = val_output.label_ids

test_logits = test_output.predictions
test_labels = test_output.label_ids

print("Validation logits shape:", val_logits.shape)
print("Validation labels shape:", val_labels.shape)
print("Test logits shape      :", test_logits.shape)
print("Test labels shape      :", test_labels.shape)

# =========================================================
# 2) Learn temperature T on validation logits only
# =========================================================
logits_tensor = torch.tensor(val_logits, dtype=torch.float32)
labels_tensor = torch.tensor(val_labels, dtype=torch.long)

temperature = nn.Parameter(torch.ones(1) * 1.0)
criterion = nn.CrossEntropyLoss()
optimizer = optim.LBFGS([temperature], lr=0.01, max_iter=50)

def closure():
    optimizer.zero_grad()
    scaled_logits = logits_tensor / temperature
    loss = criterion(scaled_logits, labels_tensor)
    loss.backward()
    return loss

before_nll = criterion(logits_tensor, labels_tensor).item()
optimizer.step(closure)
best_T = temperature.item()
after_nll = criterion(logits_tensor / temperature.detach(), labels_tensor).item()

print(f"\nLearned Temperature T = {best_T:.6f}")
print(f"NLL before scaling    = {before_nll:.6f}")
print(f"NLL after scaling     = {after_nll:.6f}")

# =========================================================
# 3) Validation probabilities before and after scaling
#    assuming class 1 is the positive class
# =========================================================
val_probs_before = F.softmax(torch.tensor(val_logits, dtype=torch.float32), dim=1)[:, 1].numpy()
val_probs_after  = F.softmax(torch.tensor(val_logits, dtype=torch.float32) / best_T, dim=1)[:, 1].numpy()

# =========================================================
# 4) Plot validation calibration curve before vs after
# =========================================================
frac_pos_before, mean_pred_before = calibration_curve(
    val_labels, val_probs_before, n_bins=10, strategy="uniform"
)
frac_pos_after, mean_pred_after = calibration_curve(
    val_labels, val_probs_after, n_bins=10, strategy="uniform"
)

plt.figure(figsize=(8, 6))
plt.plot(mean_pred_before, frac_pos_before, marker='o', label='Before Temperature Scaling')
plt.plot(mean_pred_after, frac_pos_after, marker='s', label=f'After Temperature Scaling (T={best_T:.4f})')
plt.plot([0, 1], [0, 1], '--', label='Perfect Calibration')
plt.xlabel("Mean Predicted Probability")
plt.ylabel("Fraction of Positives")
plt.title("Calibration Curve — BanglaBERT")
plt.legend()
plt.grid(True)
plt.show()

# =========================================================
# 5) Calibration metrics on validation set
# =========================================================
brier_before = brier_score_loss(val_labels, val_probs_before)
brier_after  = brier_score_loss(val_labels, val_probs_after)

logloss_before = log_loss(val_labels, val_probs_before)
logloss_after  = log_loss(val_labels, val_probs_after)

print("\nValidation Calibration Metrics")
print(f"Brier Score Before : {brier_before:.6f}")
print(f"Brier Score After  : {brier_after:.6f}")
print(f"Log Loss Before    : {logloss_before:.6f}")
print(f"Log Loss After     : {logloss_after:.6f}")

# =========================================================
# 6) Validation classification metrics before vs after
#    threshold = 0.5
# =========================================================
val_pred_before = (val_probs_before >= 0.5).astype(int)
val_pred_after  = (val_probs_after >= 0.5).astype(int)

print("\nValidation Classification Metrics (threshold = 0.5)")
print("Before Scaling")
print(f"Accuracy : {accuracy_score(val_labels, val_pred_before):.4f}")
print(f"Precision: {precision_score(val_labels, val_pred_before):.4f}")
print(f"Recall   : {recall_score(val_labels, val_pred_before):.4f}")
print(f"F1 Score : {f1_score(val_labels, val_pred_before):.4f}")

print("\nAfter Scaling")
print(f"Accuracy : {accuracy_score(val_labels, val_pred_after):.4f}")
print(f"Precision: {precision_score(val_labels, val_pred_after):.4f}")
print(f"Recall   : {recall_score(val_labels, val_pred_after):.4f}")
print(f"F1 Score : {f1_score(val_labels, val_pred_after):.4f}")

# =========================================================
# 7) Apply same T to test logits
# =========================================================
test_probs_before = F.softmax(torch.tensor(test_logits, dtype=torch.float32), dim=1)[:, 1].numpy()
test_probs_after  = F.softmax(torch.tensor(test_logits, dtype=torch.float32) / best_T, dim=1)[:, 1].numpy()

# Optional test predictions at threshold 0.5
test_pred_before = (test_probs_before >= 0.5).astype(int)
test_pred_after  = (test_probs_after >= 0.5).astype(int)

print("\nTest Classification Metrics (threshold = 0.5)")
print("Before Scaling")
print(f"Accuracy : {accuracy_score(test_labels, test_pred_before):.4f}")
print(f"Precision: {precision_score(test_labels, test_pred_before):.4f}")
print(f"Recall   : {recall_score(test_labels, test_pred_before):.4f}")
print(f"F1 Score : {f1_score(test_labels, test_pred_before):.4f}")

print("\nAfter Scaling")
print(f"Accuracy : {accuracy_score(test_labels, test_pred_after):.4f}")
print(f"Precision: {precision_score(test_labels, test_pred_after):.4f}")
print(f"Recall   : {recall_score(test_labels, test_pred_after):.4f}")
print(f"F1 Score : {f1_score(test_labels, test_pred_after):.4f}")

